In [1]:
#Importing the required packages
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
# I am Reading the File from my local and printing the top 5 and shape of the df
sales_data = pd.read_csv(r"D:\TuteDude\GenAI\MachineLearning\Assignment_14\sales_dataset.csv")
print(sales_data.head())
print("shape",sales_data.shape)

     Customer Name    State     Product  Units Sold     Unit Price  \
0              NaN   rivers    KEYBOARD        47.0  155703.699212   
1     Allison Hill    Lagos  Headphones        47.0  267992.940000   
2      Noah Rhodes  Anambra    Keyboard        47.0   42364.410000   
3  Angie Henderson    Delta    Keyboard        47.0  279444.940000   
4    Daniel Wagner    Delta      Tablet        47.0   95899.740000   

     Total Sale   Sale Date Sales Channel  \
0  7.318074e+06  2025-04-27        Online   
1  1.259567e+07  2024-03-15     Wholesale   
2  1.991127e+06  2024-12-10           NaN   
3  1.313391e+07  2024-04-05           NaN   
4  4.507288e+06  2025-01-12           NaN   

                               Order ID  price_per_unit   total_value  \
0                                   NaN   155703.699212  7.318074e+06   
1  4c636e95-025f-4543-8997-623ae0723d96   267992.940000  1.259567e+07   
2  edaf3766-1b78-4ede-9a4f-fc0c9165f2ed    42364.410000  1.991127e+06   
3  74503887-48d9

In [2]:
#Task - 1

#I am checking for the missing values and any duplicate are present in my data
print("Missing_Values",sales_data.isna().sum())
print("Duplicate_values",sales_data.duplicated().sum())
sales_data['Sales Channel'] = sales_data['Sales Channel'].fillna(sales_data['Sales Channel'].mode()[0])
sales_data['Order ID'] = sales_data['Order ID'].fillna('Unknown')
print("Missing_Values",sales_data.isna().sum())
'''Here I identified the columns as 
Units Sold → No.of units/quantity sold
Unit Price → Single Unit Price
Total Sale → revenue
Sale Date → date
Here I am creating the meaning full insights 
'''
# Price per unit
sales_data['price_per_unit'] = sales_data['Unit Price']

# Total value
sales_data['total_value'] = sales_data['Units Sold'] * sales_data['Unit Price']

# Genrerating the Revenue category based on the total value
#Total value is calculated no.of units multiply with unit price
def price_range(x):
    if x < 50000:
        return 'Low'
    elif x < 200000:
        return 'Medium'
    else:
        return 'High'

sales_data['price_range'] = sales_data['total_value'].apply(price_range)
sales_data['unit_revenue_ratio'] = sales_data['Units Sold'] / sales_data['Total Sale']
print(sales_data)


Missing_Values Customer Name        43
State                 0
Product               0
Units Sold            0
Unit Price            0
Total Sale            0
Sale Date             0
Sales Channel       106
Order ID             40
price_per_unit        0
total_value           0
revenue_category      0
year                  0
month                 0
day                   0
dtype: int64
Duplicate_values 0
Missing_Values Customer Name       43
State                0
Product              0
Units Sold           0
Unit Price           0
Total Sale           0
Sale Date            0
Sales Channel        0
Order ID             0
price_per_unit       0
total_value          0
revenue_category     0
year                 0
month                0
day                  0
dtype: int64
          Customer Name    State     Product  Units Sold     Unit Price  \
0                   NaN   rivers    KEYBOARD        47.0  155703.699212   
1          Allison Hill    Lagos  Headphones        47.0  267992.94000

In [3]:
# Task_2
#Here I am extracting the Year,Month and Day data from the sale date column into individual columns
sales_data['Sale Date'] = pd.to_datetime(sales_data['Sale Date'], errors='coerce')

# Extract features
sales_data['year'] = sales_data['Sale Date'].dt.year
sales_data['month'] = sales_data['Sale Date'].dt.month
sales_data['day'] = sales_data['Sale Date'].dt.day

# Here I extracting the length of the Product
sales_data['product_length'] = sales_data['Product'].astype(str).apply(len)
print(sales_data.head())

     Customer Name    State     Product  Units Sold     Unit Price  \
0              NaN   rivers    KEYBOARD        47.0  155703.699212   
1     Allison Hill    Lagos  Headphones        47.0  267992.940000   
2      Noah Rhodes  Anambra    Keyboard        47.0   42364.410000   
3  Angie Henderson    Delta    Keyboard        47.0  279444.940000   
4    Daniel Wagner    Delta      Tablet        47.0   95899.740000   

     Total Sale  Sale Date Sales Channel  \
0  7.318074e+06 2025-04-27        Online   
1  1.259567e+07 2024-03-15     Wholesale   
2  1.991127e+06 2024-12-10        Direct   
3  1.313391e+07 2024-04-05        Direct   
4  4.507288e+06 2025-01-12        Direct   

                               Order ID  price_per_unit   total_value  \
0                               Unknown   155703.699212  7.318074e+06   
1  4c636e95-025f-4543-8997-623ae0723d96   267992.940000  1.259567e+07   
2  edaf3766-1b78-4ede-9a4f-fc0c9165f2ed    42364.410000  1.991127e+06   
3  74503887-48d9-4846-

In [4]:
#Task -3 
#Here i am taking only the categorial data columns
categorical_cols = ['Customer Name', 'State', 'Product', 'Sales Channel']
#Applying OneHotEncoding getDummies to my Categorical columns
encoded_sales_data = pd.get_dummies(sales_data, columns=categorical_cols)
#Displaying the Transformed columns
encoded_sales_data.head()

,Units Sold,Unit Price,Total Sale,Sale Date,Order ID,price_per_unit,total_value,revenue_category,year,month,...,Product_MONITOR,Product_Monitor,Product_PHONE,Product_Phone,Product_TABLET,Product_Tablet,Sales Channel_Direct,Sales Channel_Online,Sales Channel_Retail,Sales Channel_Wholesale
0,47.0,155703.699212,7.318074e+06,2025-04-27,Unknown,155703.699212,7.318074e+06,High,2025,4,...,False,False,False,False,False,False,False,True,False,False
1,47.0,267992.940000,1.259567e+07,2024-03-15,4c636e95-025f-4543-8997-623ae0723d96,267992.940000,1.259567e+07,High,2024,3,...,False,False,False,False,False,False,False,False,False,True
2,47.0,42364.410000,1.991127e+06,2024-12-10,edaf3766-1b78-4ede-9a4f-fc0c9165f2ed,42364.410000,1.991127e+06,High,2024,12,...,False,False,False,False,False,False,True,False,False,False
3,47.0,279444.940000,1.313391e+07,2024-04-05,74503887-48d9-4846-95c5-51fcfba57cc8,279444.940000,1.313391e+07,High,2024,4,...,False,False,False,False,False,False,True,False,False,False
4,47.0,95899.740000,4.507288e+06,2025-01-12,8639bd41-8b15-4d94-a42d-0cd7fd359f6a,95899.740000,4.507288e+06,High,2025,1,...,False,False,False,False,False,True,True,False,False,False


In [5]:
#Task - 4
print("shape",sales_data.shape)
#Here i am seperating the columns by categorical and numerical
categorical_cols = ['Customer Name', 'State', 'Product', 'Sales Channel']
numerical_cols = ['Units Sold', 'Unit Price', 'Total Sale']
#Here I did the column transformed 
col_trans = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=False), categorical_cols)],remainder='passthrough')

sales_data_trans = col_trans.fit_transform(sales_data)
print(sales_data_trans)

shape (550, 18)
[[0.0 0.0 0.0 ... 'High' 6.422454990216135e-06 8]
 [0.0 0.0 0.0 ... 'High' 3.7314415820058545e-06 10]
 [0.0 0.0 0.0 ... 'High' 2.3604719149871313e-05 8]
 ...
 [0.0 0.0 0.0 ... 'High' 9.607539689707212e-06 5]
 [0.0 0.0 0.0 ... 'High' 3.3605662419695066e-05 7]
 [0.0 0.0 0.0 ... 'High' 1.4280179696069221e-05 6]]


In [6]:
#Task - 5
#I am Peroforming the Standard Scalar Feature scaling for numeric columns
scaler = StandardScaler()
numerical_cols = ['Units Sold', 'Unit Price', 'Total Sale']
sales_data[numerical_cols] = scaler.fit_transform(sales_data[numerical_cols])
sales_data[numerical_cols].head()
categorical_cols = ['Customer Name', 'State', 'Product', 'Sales Channel']
numerical_cols = ['Units Sold', 'Unit Price', 'Total Sale']
std_scalar = StandardScaler().fit_transform(sales_data[numerical_cols])
minmax_scalar = MinMaxScaler().fit_transform(sales_data[numerical_cols])

print("Original_Sales_Data:", sales_data)
print("StandardScaler_Sales_Data:", pd.DataFrame(std_scalar, columns=numerical_cols))
print("MinMaxScaler_Sales_Data:", pd.DataFrame(minmax_scalar, columns=numerical_cols))

Original_Sales_Data:           Customer Name    State     Product  Units Sold    Unit Price  \
0                   NaN   rivers    KEYBOARD    0.002802  3.636673e-16   
1          Allison Hill    Lagos  Headphones    0.002802  1.403112e+00   
2           Noah Rhodes  Anambra    Keyboard    0.002802 -1.416233e+00   
3       Angie Henderson    Delta    Keyboard    0.002802  1.546210e+00   
4         Daniel Wagner    Delta      Tablet    0.002802 -7.472812e-01   
..                  ...      ...         ...         ...           ...   
545    Zachary Mitchell    Ekiti     Monitor   -2.869755 -1.645842e+00   
546                 NaN  Katsina      Tablet    0.002802  8.504515e-01   
547  Katherine Martinez   Bauchi       Phone    0.002802 -6.450032e-01   
548          Jodi Roach    Niger     Charger    0.002802 -1.573770e+00   
549     Brandon Fleming      imo      Camera    0.002802 -1.070573e+00   

     Total Sale  Sale Date Sales Channel  \
0     -0.003835 2025-04-27        Online   
1 

Task - 5 Explanation: 
StandardScaler standardizes numerical features so mean becomes nearly 0 and its standard deviation becomes nearly 1. It will helps to improve the model performance with all numeric features.

In [7]:
#Task - 6
#Here I am implementing with MinMaxScalar
scaler = MinMaxScaler()

sales_data[numerical_cols] = scaler.fit_transform(sales_data[numerical_cols])

sales_data[numerical_cols].head()

,Units Sold,Unit Price,Total Sale
0,0.464646,0.517727,0.253174
1,0.464646,0.894493,0.437211
2,0.464646,0.137438,0.067415
3,0.464646,0.932918,0.455981
4,0.464646,0.317066,0.155157


Task - 6 Explaination
MinMaxScaler transforms numerical features to a fixed range between 0 and 1. Unlike StandardScaler, it does not center the data but rescales it, making it useful when a bounded range is required.


In [8]:
#Task -7
# I am Reading the File from my local and printing the top 5 and shape of the df
sales_data = pd.read_csv(r"D:\TuteDude\GenAI\MachineLearning\Assignment_14\sales_dataset.csv")
print(sales_data.head())
print("shape",sales_data.shape)
categorical_cols = ['Customer Name', 'State', 'Product', 'Sales Channel']
numerical_cols = ['Units Sold', 'Unit Price', 'Total Sale']

process_pipeline = ColumnTransformer(transformers=[('num', Pipeline(steps=[('scaler', StandardScaler())]), numerical_cols),
                                               ('cat', Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)])

sales_data_processed = process_pipeline.fit_transform(sales_data)

print(sales_data_processed)
   

     Customer Name    State     Product  Units Sold     Unit Price  \
0              NaN   rivers    KEYBOARD        47.0  155703.699212   
1     Allison Hill    Lagos  Headphones        47.0  267992.940000   
2      Noah Rhodes  Anambra    Keyboard        47.0   42364.410000   
3  Angie Henderson    Delta    Keyboard        47.0  279444.940000   
4    Daniel Wagner    Delta      Tablet        47.0   95899.740000   

     Total Sale   Sale Date Sales Channel  \
0  7.318074e+06  2025-04-27        Online   
1  1.259567e+07  2024-03-15     Wholesale   
2  1.991127e+06  2024-12-10           NaN   
3  1.313391e+07  2024-04-05           NaN   
4  4.507288e+06  2025-01-12           NaN   

                               Order ID  price_per_unit   total_value  \
0                                   NaN   155703.699212  7.318074e+06   
1  4c636e95-025f-4543-8997-623ae0723d96   267992.940000  1.259567e+07   
2  edaf3766-1b78-4ede-9a4f-fc0c9165f2ed    42364.410000  1.991127e+06   
3  74503887-48d9

In [10]:
#Task - 8
X = sales_data.iloc[:,:-1]
Y = sales_data['Total Sale']
categorical_cols = ['Customer Name', 'State', 'Product', 'Sales Channel']
numerical_cols = ['Units Sold', 'Unit Price', 'Total Sale']

process_pipeline = ColumnTransformer(transformers=[('num', Pipeline(steps=[('scaler', StandardScaler())]), numerical_cols),
                                               ('cat', Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)])

model_pipeline = Pipeline([('preprocessing', process_pipeline),('model', LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print(y_pred[:5])
sales_data_processed = process_pipeline.fit_transform(sales_data)

r2 = r2_score(y_test, y_pred)
mae_score = mean_absolute_error(y_test, y_pred)
rmse_score = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2 Score:", r2)
print("MAE Score:", mae_score)
print("RMSE Score:", rmse_score)

[ 7049959.03781467  6906561.68092467 12782806.74212838   -30580.48414035
  5865958.43312882]
R2 Score: 0.9999849102379884
MAE Score: 9039.747070193496
RMSE Score: 19245.731380736866


Task - 9
1. Pipelines are important because they automate the preprocessing and modeling steps, ensuring consistency and reducing manual errors.

2. Pipelines solve problems such as data leakage, inconsistent preprocessing, and complex workflows. They ensure that the same transformations are applied to both training and testing data.

3. Manual preprocessing involves applying transformations step-by-step, which can lead to errors and inconsistencies. Pipeline-based preprocessing integrates all steps into a single workflow, making it more reliable, scalable, and easier to manage.
